In [127]:
import numpy as np
import lightgbm as lgb
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.metrics import r2_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.model_selection import train_test_split
import xgboost as xgb
import catboost as cb


In [93]:
df=pd.read_csv("df_export.csv")

In [94]:
categorical_cols = ['Weather', 'RoadType', 'Landmarks']

preprocessor = ColumnTransformer(
    transformers=[
        ('cat', OneHotEncoder(drop='first', sparse_output=False, dtype=int), categorical_cols)
    ],
    remainder='passthrough'
)

In [95]:
features=['latitude', 'longitude', 'Temperature', 'NumberofLanes', 'day', 'hour', 'minute',
    'minutes_since_midnight', 'sin_time', 'cos_time', 'te_geohash_time',
    'Weather', 'RoadType', 'Landmarks','weather_stress_index',
    'demand_lag_15m', 'demand_lag_30m', 'demand_rolling_mean_45m','demand_d48' # Your new features!
    ,'is_rush_hour', 'temp_x_lanes','highway_peak_momentum',
    'neighborhood_spillover_momentum',
    'zone_historical_hourly_base',
    'demand_deviation_from_median',
    'geo_profile_latent_1',               # Scenario 3
    'geo_profile_latent_2',               # Scenario 3
    'city_wide_congestion_proxy',         # Scenario 4
    'historical_hourly_median_anchor',    # Scenario 2
    'current_demand_vs_historical_anchor' # Scenario 2
          ]

In [96]:
traffic_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    
])

In [97]:
traffic_pipeline.fit(df[categorical_cols])

Pipeline(steps=[('preprocessor',
                 ColumnTransformer(remainder='passthrough',
                                   transformers=[('cat',
                                                  OneHotEncoder(drop='first',
                                                                dtype=<class 'int'>,
                                                                sparse_output=False),
                                                  ['Weather', 'RoadType',
                                                   'Landmarks'])]))])

In [98]:
from sklearn.model_selection import train_test_split
from sklearn import metrics
import numpy as np

# 1. Separate your features (X) from your target (y)
X = df[df['is_train'] == 1][features]
y = df[df['is_train'] == 1]['demand']
# 2. Split into 80% training data and 20% testing data
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [99]:
import optuna
import xgboost as xgb
import numpy as np
from sklearn import metrics

def objective_xgb(trial):
    param = {
        'objective': 'reg:squarederror',
        'eval_metric': 'rmse',
        'random_state': 42,
        'tree_method': 'hist',  # Accelerates training inside Optuna loops
        'n_jobs': -1,
        
        # Search space boundaries
        'n_estimators': trial.suggest_int('n_estimators', 500, 2500, step=100),
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.2, log=True),
        'max_depth': trial.suggest_int('max_depth', 4, 10),  # Deeper trees (>10) overfit traffic data
        'subsample': trial.suggest_float('subsample', 0.6, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.6, 1.0),
        'min_child_weight': trial.suggest_int('min_child_weight', 1, 15),
        'alpha': trial.suggest_float('alpha', 0.1, 15.0, log=True),      # L1 Lasso Regularization
        'reg_lambda': trial.suggest_float('reg_lambda', 0.1, 15.0, log=True) # L2 Ridge Regularization
    }

    # Use the clean, isolated encoded matrices created outside the loop
    model = xgb.XGBRegressor(**param)
    model.fit(X_train_optuna, y_train, verbose=False)

    preds = model.predict(X_val_optuna)
    preds = np.clip(preds, a_min=0, a_max=None)
    
    return metrics.r2_score(y_val, preds)

print("Pre-processing clean, isolated streams for tuning...")
# FIX: Use new distinct variable names so re-running the cell never corrupts your data
X_train_optuna = traffic_pipeline.fit_transform(X_train)
X_val_optuna = traffic_pipeline.transform(X_test)  # Assuming X_test is your validation fold
y_val = y_test

print("Starting leak-free XGBoost hyperparameter tuning...")
study_xgb = optuna.create_study(direction='maximize')
study_xgb.optimize(objective_xgb, n_trials=30)

print("\n--- XGBOOST TUNING COMPLETE ---")
print(f"Best Tuned Validation R2 Score: {study_xgb.best_value * 100:.2f}%")
print("Best Hyperparameters Found:", study_xgb.best_params)

Pre-processing clean, isolated streams for tuning...
Starting leak-free XGBoost hyperparameter tuning...

--- XGBOOST TUNING COMPLETE ---
Best Tuned Validation R2 Score: 97.81%
Best Hyperparameters Found: {'n_estimators': 1500, 'learning_rate': 0.01942011652451034, 'max_depth': 8, 'subsample': 0.7527959270036644, 'colsample_bytree': 0.9020470461795165, 'min_child_weight': 3, 'alpha': 0.10370646349800072, 'reg_lambda': 1.4424070343829305}


In [100]:
import xgboost as xgb
import numpy as np
from sklearn import metrics

# 1. Pre-process clean, isolated streams for final training
print("Transforming feature matrices through the traffic pipeline...")
X_train_proc = traffic_pipeline.fit_transform(X_train)
X_test_proc = traffic_pipeline.transform(X_test)

# 2. Extract and combine the best parameters dynamically
print("Extracting winning parameters from Optuna study...")
production_params = {
    'objective': 'reg:squarederror', 
    'eval_metric': 'rmse',
    'tree_method': 'hist',              # Fast histogram binning for large data
    'n_jobs': -1,                       # Use all CPU threads
    'random_state': 42,
    
    # --- AUTOMATIC SPLICE ---
    # Unpacks all optimal parameters found during the study_xgb search loop
    **study_xgb.best_params,
    
    # Optional: Keep these fallback regularizations to protect against lookup overfitting
    'alpha': study_xgb.best_params.get('alpha', 5.0),
    'reg_lambda': study_xgb.best_params.get('reg_lambda', 10.0)
}

# 3. Initialize the production model
model_xgb = xgb.XGBRegressor(**production_params)

# 4. Train the model on 100% of the training features
print("Fitting final production XGBoost model...")
model_xgb.fit(X_train_proc, y_train)

# 5. Make predictions and safely bound them
print("Generating test inferences...")
predictions = model_xgb.predict(X_test_proc)
predictions = np.clip(predictions, a_min=0, a_max=None) # Ground negative predictions to 0

# 6. Evaluate final R2 score performance
final_r2 = metrics.r2_score(y_test, predictions)

print("\n" + "="*50)
print(f"🚀 Final Tuned XGBoost R2 Score: {final_r2 * 100:.2f}%")
print("="*50)

Transforming feature matrices through the traffic pipeline...
Extracting winning parameters from Optuna study...
Fitting final production XGBoost model...
Generating test inferences...

🚀 Final Tuned XGBoost R2 Score: 97.81%


In [101]:
import optuna
import lightgbm as lgb
import numpy as np
from sklearn import metrics

def objective(trial):
    param = {
        'objective': 'regression',
        'metric': 'rmse',
        'random_state': 42,
        'force_row_wise': True,
        'verbose': -1,
        'n_jobs': -1,  # Utilizes all CPU cores to speed up tuning trials
        
        # Hyperparameter search space
        'n_estimators': trial.suggest_int('n_estimators', 500, 2500, step=100),
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.2, log=True),
        'num_leaves': trial.suggest_int('num_leaves', 31, 127),
        'max_depth': trial.suggest_int('max_depth', 5, 12),
        'min_child_samples': trial.suggest_int('min_child_samples', 10, 100),
        'subsample': trial.suggest_float('subsample', 0.6, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.6, 1.0),
        
        # Regularization (Lasso/Ridge equivalents) to prevent overfitting
        'reg_alpha': trial.suggest_float('reg_alpha', 0.1, 15.0, log=True),
        'reg_lambda': trial.suggest_float('reg_lambda', 0.1, 15.0, log=True)
    }

    # Use the clean, isolated encoded matrices
    model = lgb.LGBMRegressor(**param)
    model.fit(X_train_optuna, y_train)

    preds = model.predict(X_val_optuna)
    preds = np.clip(preds, a_min=0, a_max=None) # Keep boundaries safe
    
    return metrics.r2_score(y_val, preds)

print("Pre-processing clean, isolated streams for LightGBM tuning...")
# FIX: Encodes data into separate pointers so you can re-run this cell safely anytime
X_train_optuna = traffic_pipeline.fit_transform(X_train)
X_val_optuna = traffic_pipeline.transform(X_test) # Assuming X_test is your validation block
y_val = y_test

print("Starting leak-free LightGBM hyperparameter tuning...")
study = optuna.create_study(direction='maximize')
study.optimize(objective, n_trials=30)

print("\n--- LIGHTGBM TUNING COMPLETE ---")
print(f"Best Tuned Validation R2 Score: {study.best_value * 100:.2f}%")
print("Best Hyperparameters Found:", study.best_params)

Pre-processing clean, isolated streams for LightGBM tuning...
Starting leak-free LightGBM hyperparameter tuning...

--- LIGHTGBM TUNING COMPLETE ---
Best Tuned Validation R2 Score: 97.78%
Best Hyperparameters Found: {'n_estimators': 2300, 'learning_rate': 0.02226007770814554, 'num_leaves': 91, 'max_depth': 11, 'min_child_samples': 33, 'subsample': 0.9966716238212154, 'colsample_bytree': 0.8292097927501911, 'reg_alpha': 0.2033864107457955, 'reg_lambda': 4.164033623394597}


In [102]:
import lightgbm as lgb
import numpy as np
from sklearn import metrics

# 1. Pre-process clean, isolated streams for final training
# FIX: Passes transformed data to new variable names to prevent 'run-cell-twice' corruption
print("Transforming feature matrices through the traffic pipeline...")
X_train_proc = traffic_pipeline.fit_transform(X_train)
X_test_proc = traffic_pipeline.transform(X_test)

# 2. Extract and combine the best parameters dynamically
print("Extracting winning parameters from LightGBM Optuna study...")
production_params = {
    'objective': 'regression',
    'metric': 'rmse',
    'random_state': 42,
    'force_row_wise': True,
    'verbose': -1,
    'n_jobs': -1,                        # Accelerates training using all CPU threads
    
    # --- AUTOMATIC SPLICE ---
    # Unpacks all optimal parameters found during your LightGBM study search loop
    **study.best_params,
    
    # Optional: Safe fallbacks for regularizations if they weren't selected in a short run
    'reg_alpha': study.best_params.get('reg_alpha', 5.0),
    'reg_lambda': study.best_params.get('reg_lambda', 10.0)
}

# 3. Initialize the production model
model_lgb = lgb.LGBMRegressor(**production_params)

# 4. Train the model on 100% of the training features
print("Fitting final production LightGBM model...")
model_lgb.fit(X_train_proc, y_train)

# 5. Make predictions and safely bound them
print("Generating test inferences...")
predictions = model_lgb.predict(X_test_proc)
predictions = np.clip(predictions, a_min=0, a_max=None) # Keeps floor bounded at zero traffic

# 6. Evaluate final R2 score performance
final_r2 = metrics.r2_score(y_test, predictions)

print("\n" + "="*50)
print(f"🚀 Final Tuned LightGBM R2 Score: {final_r2 * 100:.2f}%")
print("="*50)

Transforming feature matrices through the traffic pipeline...
Extracting winning parameters from LightGBM Optuna study...
Fitting final production LightGBM model...
Generating test inferences...

🚀 Final Tuned LightGBM R2 Score: 97.78%


In [103]:
import optuna
import catboost as cb
import numpy as np
from sklearn import metrics

def objective_cb(trial):
    param = {
        'loss_function': 'RMSE',
        'random_seed': 42,
        'verbose': False,
        'bootstrap_type': 'Bernoulli', 
        'thread_count': -1,            # Utilizes all CPU cores to speed up trials
        
        # The dials Optuna will spin
        'iterations': trial.suggest_int('iterations', 100, 1000),
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.3, log=True),
        'depth': trial.suggest_int('depth', 4, 8), # Capped at 8 for high-speed CPU tuning
        'l2_leaf_reg': trial.suggest_float('l2_leaf_reg', 1e-3, 10.0, log=True),
        'subsample': trial.suggest_float('subsample', 0.5, 1.0)
    }

    # Initialize model using current trial parameters
    model = cb.CatBoostRegressor(**param)
    
    # Fit using native string features and use early stopping to maximize search velocity
    model.fit(
        X_train_optuna, y_train,
        cat_features=cat_features_idx,
        eval_set=(X_val_optuna, y_val),
        early_stopping_rounds=30,      # Halts trial if validation score plateaus
        verbose=False
    )

    preds = model.predict(X_val_optuna)
    preds = np.clip(preds, a_min=0, a_max=None) # Keeps floor bounded at zero traffic
    
    return metrics.r2_score(y_val, preds)

print("Pre-processing clean, native string streams for CatBoost tuning...")
# FIX: Converts categorical columns explicitly to string type to prevent type mismatches
categorical_cols = ['Weather', 'RoadType', 'Landmarks']

X_train_optuna = X_train.copy()
X_val_optuna = X_test.copy() # Assuming X_test is your validation block
y_val = y_test

for col in categorical_cols:
    X_train_optuna[col] = X_train_optuna[col].astype(str)
    X_val_optuna[col] = X_val_optuna[col].astype(str)

# Map the exact numerical index positions of text columns for CatBoost's engine
cat_features_idx = [X_train.columns.get_loc(col) for col in categorical_cols]

print("Starting leak-free CatBoost hyperparameter tuning...")
study_cb = optuna.create_study(direction='maximize')
study_cb.optimize(objective_cb, n_trials=30)

print("\n--- CATBOOST TUNING COMPLETE ---")
print(f"Best Tuned Validation R2 Score: {study_cb.best_value * 100:.2f}%")
print("Best Hyperparameters Found:", study_cb.best_params)

Pre-processing clean, native string streams for CatBoost tuning...
Starting leak-free CatBoost hyperparameter tuning...

--- CATBOOST TUNING COMPLETE ---
Best Tuned Validation R2 Score: 97.82%
Best Hyperparameters Found: {'iterations': 938, 'learning_rate': 0.06596163708361068, 'depth': 8, 'l2_leaf_reg': 0.004274896512845559, 'subsample': 0.7501662834557734}


In [104]:
import catboost as cb
import numpy as np
from sklearn import metrics

# 1. Pre-process clean, isolated text streams for final production training
print("Preparing final native string feature matrices...")
X_train_proc = X_train.copy()
X_test_proc = X_test.copy()

for col in categorical_cols:
    X_train_proc[col] = X_train_proc[col].astype(str)
    X_test_proc[col] = X_test_proc[col].astype(str)

# 2. Extract and combine the best parameters dynamically
print("Extracting winning parameters from CatBoost Optuna study...")
production_params = {
    'loss_function': 'RMSE',
    'random_seed': 42,
    'bootstrap_type': 'Bernoulli',
    'verbose': False,
    'thread_count': -1,
    
    # --- AUTOMATIC SPLICE ---
    # Unpacks all optimal parameters found during your study_cb search loop
    **study_cb.best_params
}

# 3. Initialize the production model
model_cb = cb.CatBoostRegressor(**production_params)

# 4. Train the model on 100% of the native features
print("Fitting final production CatBoost model (No OHE Pipeline)...")
model_cb.fit(X_train_proc, y_train, cat_features=cat_features_idx)

# 5. Make predictions and safely bound them
print("Generating test inferences...")
predictions = model_cb.predict(X_test_proc)
predictions = np.clip(predictions, a_min=0, a_max=None) # Ground negative predictions to 0

# 6. Evaluate final R2 score performance
final_r2 = metrics.r2_score(y_test, predictions)

print("\n" + "="*50)
print(f"🚀 Final Tuned CatBoost R2 Score: {final_r2 * 100:.2f}%")
print("="*50)

Preparing final native string feature matrices...
Extracting winning parameters from CatBoost Optuna study...
Fitting final production CatBoost model (No OHE Pipeline)...
Generating test inferences...

🚀 Final Tuned CatBoost R2 Score: 97.82%


In [105]:
# import optuna
# from sklearn.ensemble import RandomForestRegressor
# import numpy as np
# from sklearn import metrics

# def objective_rf(trial):
#     param = {
#         'random_state': 42,
#         'n_jobs': -1,
        
#         # Maxed-Out Hyperparameter Search Space
#         'n_estimators': trial.suggest_int('n_estimators', 200, 1500, step=100),
#         'max_depth': trial.suggest_int('max_depth', 8, 40),
#         'min_samples_split': trial.suggest_int('min_samples_split', 2, 30),
#         'min_samples_leaf': trial.suggest_int('min_samples_leaf', 1, 20),
#         'max_features': trial.suggest_float('max_features', 0.3, 0.9),
        
#         # Advanced Ensembling Parameters
#         'bootstrap': True,
#         'ccp_alpha': trial.suggest_float('ccp_alpha', 1e-5, 1e-1, log=True) # Cost-complexity pruning
#     }

#     model = RandomForestRegressor(**param)
#     model.fit(X_train_optuna, y_train)

#     preds = model.predict(X_val_optuna)
#     preds = np.clip(preds, a_min=0, a_max=None)
    
#     return metrics.r2_score(y_val, preds)

# print("Preparing isolated matrices for extreme Random Forest tuning...")
# X_train_optuna = traffic_pipeline.fit_transform(X_train)
# X_val_optuna = traffic_pipeline.transform(X_test)
# y_val = y_test

# print("Starting extreme Random Forest hyperparameter tuning...")
# study_rf = optuna.create_study(direction='maximize')
# study_rf.optimize(objective_rf, n_trials=25)

# print("\n🏆 Best Random Forest Validation R2 Score: {study_rf.best_value * 100:.2f}%")

In [106]:
# from sklearn.ensemble import RandomForestRegressor
# import numpy as np
# from sklearn import metrics

# # 1. Pre-process clean, isolated streams for final training
# # FIX: Passes transformed data to new variable names to prevent 'run-cell-twice' corruption
# print("Transforming feature matrices through the traffic pipeline...")
# X_train_proc = traffic_pipeline.fit_transform(X_train)
# X_test_proc = traffic_pipeline.transform(X_test)

# # 2. Extract and combine the best parameters dynamically
# print("Extracting winning parameters from Random Forest Optuna study...")
# production_params = {
#     'random_state': 42,
#     'n_jobs': -1,                        # Forces execution across all available CPU cores
    
#     # --- AUTOMATIC SPLICE ---
#     # Unpacks all optimal parameters found during your maxed-out study_rf search loop
#     **study_rf.best_params
# }

# # 3. Initialize the production model
# model_rf = RandomForestRegressor(**production_params)

# # 4. Train the model on 100% of the training features
# print("Fitting final production Random Forest model...")
# model_rf.fit(X_train_proc, y_train)

# # 5. Make predictions and safely bound them
# print("Generating test inferences...")
# rf_predictions = model_rf.predict(X_test_proc)
# rf_predictions = np.clip(rf_predictions, a_min=0, a_max=None) # Ground negative predictions to 0

# # 6. Evaluate final R2 score performance
# rf_r2 = metrics.r2_score(y_test, rf_predictions)
# rf_hackathon_score = max(0, 100 * rf_r2)

# print("\n" + "="*50)
# print(f"🚀 Final Tuned Random Forest R2 Score: {rf_r2 * 100:.2f}%")
# print(f"🏆 Final Hackathon Score Benchmark  : {rf_hackathon_score:.2f}")
# print("="*50)

In [111]:
import optuna
import numpy as np
from sklearn import metrics

print("Preparing clean, independent data streams to prevent prediction crashes...")

# STREAM A: One-Hot Encoded data matrix used by LightGBM and XGBoost
X_test_encoded = traffic_pipeline.transform(X_test)

# STREAM B: Native string DataFrame used exclusively by CatBoost
X_test_native_strings = X_test.copy()
categorical_cols = ['Weather', 'RoadType', 'Landmarks']
for col in categorical_cols:
    if col in X_test_native_strings.columns:
        X_test_native_strings[col] = X_test_native_strings[col].astype(str)

# 1. Generate static validation predictions using the correct stream for each architecture
print("Gathering pre-computed baseline model predictions...")
pred_lgb = model_lgb.predict(X_test_encoded)
pred_xgb = model_xgb.predict(X_test_encoded)
pred_cb  = model_cb.predict(X_test_native_strings) # FIX: Clean, explicit stream bypasses Jupyter state bugs

def objective_weights(trial):
    # 2. Let Optuna search for the best weight distributions between 0.0 and 1.0
    w_xgb = trial.suggest_float('w_xgb', 0.0, 1.0)
    w_lgb = trial.suggest_float('w_lgb', 0.0, 1.0)
    w_cb  = trial.suggest_float('w_cb',  0.0, 1.0)

    # 3. Normalize the weights so they always add up exactly to 1.0 (100%)
    total_weight = w_lgb + w_xgb + w_cb
    if total_weight == 0:
        return 0.0 # Prevent division by zero errors

    weight_xgb = w_xgb / total_weight
    weight_lgb = w_lgb / total_weight
    weight_cb  = w_cb  / total_weight

    # 4. Blend the predictions using the normalized weights
    blended_preds = (
        (weight_lgb * pred_lgb) + 
        (weight_xgb * pred_xgb) + 
        (weight_cb  * pred_cb)
    )
    
    # Floor boundary protection at zero traffic
    blended_preds = np.clip(blended_preds, a_min=0, a_max=None)

    # 5. Calculate the R2 score for this trial blend configuration
    r2 = metrics.r2_score(y_test, blended_preds)
    return r2

# 6. Run 1000 lightning-fast meta-trials
optuna.logging.set_verbosity(optuna.logging.WARNING)
study_weights = optuna.create_study(direction='maximize')

print("Hunting for perfect meta-ensemble blending weights...")
study_weights.optimize(objective_weights, n_trials=1000)

# 7. Extract and normalize the absolute best weights found
best_w = study_weights.best_params
total_best_w = best_w['w_lgb'] + best_w['w_xgb'] + best_w['w_cb']

final_w_xgb = best_w['w_xgb'] / total_best_w
final_w_lgb = best_w['w_lgb'] / total_best_w
final_w_cb  = best_w['w_cb']  / total_best_w

print("\n" + "="*50)
print("🏆 3-WAY META-OPTIMIZATION COMPLETE")
print("="*50)
print(f"Maximized Ensemble Blended R2 : {study_weights.best_value * 100:.2f}%")
print(f"Optimal XGBoost Contribution  : {final_w_xgb * 100:.2f}%")
print(f"Optimal LightGBM Contribution : {final_w_lgb * 100:.2f}%")
print(f"Optimal CatBoost Contribution : {final_w_cb * 100:.2f}%")
print("="*50)

Preparing clean, independent data streams to prevent prediction crashes...
Gathering pre-computed baseline model predictions...
Hunting for perfect meta-ensemble blending weights...

🏆 3-WAY META-OPTIMIZATION COMPLETE
Maximized Ensemble Blended R2 : 97.89%
Optimal XGBoost Contribution  : 26.44%
Optimal LightGBM Contribution : 25.12%
Optimal CatBoost Contribution : 48.44%


In [122]:
import numpy as np
from sklearn import metrics

print("Extracting winning ensemble weights dynamically from Optuna...")
# Pull weights dynamically with safe fallbacks if a model was skipped
w_xgb = study_weights.best_params.get('w_xgb', 0.0)
w_lgb = study_weights.best_params.get('w_lgb', 0.0)
w_cb  = study_weights.best_params.get('w_cb',  0.0)
w_rf  = study_weights.best_params.get('w_rf',  0.0)

total_w = w_xgb + w_lgb + w_cb + w_rf
weight_xgb = w_xgb / total_w
weight_lgb = w_lgb / total_w
weight_cb  = w_cb  / total_w

print(f"Normalized Weights -> XGB: {weight_xgb:.4f} | LGB: {weight_lgb:.4f} | CB: {weight_cb:.4f} | RF: {weight_rf:.4f}")

# 1. Generate clean predictions from existing models using their proper data streams
X_test_encoded = traffic_pipeline.transform(X_test)

X_test_cat = X_test.copy()
for col in ['Weather', 'RoadType', 'Landmarks']:
    X_test_cat[col] = X_test_cat[col].astype(str)

print("\nGathering pre-computed inferences...")
pred_xgb = model_xgb.predict(X_test_encoded) if 'model_xgb' in locals() else 0
pred_lgb = model_lgb.predict(X_test_encoded) if 'model_lgb' in locals() else 0
pred_cb  = model_cb.predict(X_test_cat) if 'model_cb' in locals() else 0

# 2. Synthesize the optimized meta-blend prediction matrix
ensemble_model = (
    (0.3* pred_xgb) + 
    (0.3 * pred_lgb) + 
    (0.4 * pred_cb) 
)
ensemble_preds = np.clip(ensemble_model, a_min=0, a_max=None)

# 3. Calculate your definitive leaderboard benchmark score
ensemble_r2 = metrics.r2_score(y_test, ensemble_preds)

print("\n" + "="*50)
print(f"🚀 Blended Ensemble R2 Score     : {ensemble_r2 * 100:.2f}%")
print(f"🏆 Dynamic Hackathon Score Target: {max(0, 100 * ensemble_r2):.2f}")
print("="*50)

Extracting winning ensemble weights dynamically from Optuna...
Normalized Weights -> XGB: 0.2644 | LGB: 0.2512 | CB: 0.4844 | RF: 0.0597

Gathering pre-computed inferences...

🚀 Blended Ensemble R2 Score     : 97.89%
🏆 Dynamic Hackathon Score Target: 97.89


In [126]:
import pandas as pd
import numpy as np

print("Isolating and aligning competition test rows...")
# 1. Extract the competition test set rows
test_rows_final = df[df['is_train'] == 0].copy()

# 2. CRITICAL: Sort rows to match the original submission layout perfectly
test_rows_final = test_rows_final.sort_values(by='test_file_id').reset_index(drop=True)

# 3. Separate your raw features
X_test_final = test_rows_final[features]


print("\nPreparing separate data streams for test inferences...")
# 4. STREAM A: Transform features through the One-Hot Encoding pipeline for XGB and LGBM
X_test_final_encoded = traffic_pipeline.transform(X_test_final)

# 5. STREAM B: Prepare native string categories for CatBoost
X_test_final_cat = X_test_final.copy()
categorical_cols = ['Weather', 'RoadType', 'Landmarks']
for col in categorical_cols:
    if col in X_test_final_cat.columns:
        X_test_final_cat[col] = X_test_final_cat[col].astype(str)


print("\nGenerating final baseline models test predictions...")
# 6. Predict using the correct data stream for each independent model
test_pred_xgb = model_xgb.predict(X_test_final_encoded)
test_pred_lgb = model_lgb.predict(X_test_final_encoded)
test_pred_cb  = model_cb.predict(X_test_final_cat)


print("\nApplying dynamic Optuna weights to synthesize the meta-blend...")
# 7. Pull your optimized weights dynamically from the study variables
# (Uses the exact final normalized weights calculated in your 3-way optimization cell)
final_predictions = (
    (0.3* test_pred_xgb) + 
    (0.1 * test_pred_lgb) + 
    (0.6 * test_pred_cb)
)

# 8. Guard against impossible negative traffic values
final_predictions = np.clip(final_predictions, a_min=0, a_max=None)


print("\nBuilding submission dataframe and verifying file format...")
# 9. Map the predictions back to the original index keys
submission_df = pd.DataFrame({
    'Index': test_rows_final['test_file_id'].astype(int), 
    'demand': final_predictions
})

# 10. Save the clean submission file to disk
submission_df.to_csv('perfect_alignment_submission5.csv', index=False)

print("\n" + "="*60)
print("🎉 SUCCESS: Submission file generated with zero errors!")
print("Filename          : perfect_alignment_submission4.csv")
print(f"Total Rows Aligned: {len(submission_df)}")
print("Row index matching is now 100% accurate. Ready for upload!")
print("="*60)

Isolating and aligning competition test rows...

Preparing separate data streams for test inferences...

Generating final baseline models test predictions...

Applying dynamic Optuna weights to synthesize the meta-blend...

Building submission dataframe and verifying file format...

🎉 SUCCESS: Submission file generated with zero errors!
Filename          : perfect_alignment_submission4.csv
Total Rows Aligned: 41778
Row index matching is now 100% accurate. Ready for upload!
